# Seeing ViT Registers — Base, Large and Giant

Independent inference reproduction of [Vision Transformers Need Registers](https://arxiv.org/abs/2309.16588).
All six pinned models use the same 20 images, seed 42, 518×518 inputs and float32 settings.
The implementation lives in `src/`; `config.yaml` is the settings source.

**Colab:** select a GPU runtime with sufficient memory; Giant was tested on an A100 40 GB, not smaller GPUs.
**Local:** install `requirements.txt` and start from the repository root. Initial model downloads total about 12 GB.
This notebook recreates every saved figure and table, including both feature spaces, prefix tokens, and the image bootstrap.

In [ ]:
# Environment setup only; the experiment itself is implemented in src/.
from pathlib import Path
import os
import subprocess
import sys
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path('/content/seeing-vit-registers')
    if not (checkout / 'src/pipeline.py').exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/turhancan97/seeing-vit-registers.git', str(checkout)], check=True)
    os.chdir(checkout)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
assert Path('src/pipeline.py').exists(), 'Start the local notebook from the repository root.'
sys.path.insert(0, str(Path.cwd()))

## Recompute the complete study

Fresh inference is the default. CLS and four register tokens are measured separately and removed before
any 37×37 patch-grid reshape. Output provenance records all revisions, versions, settings and source hashes.

In [ ]:
from src.pipeline import run_all
from src.notebook import show_results, show_figures
results = run_all()

## Both feature spaces

`raw` means `hidden_states[-1]` before the final LayerNorm. `normalized` means `last_hidden_state` after
that learned normalization. Every forward verifies the relation. Intermediate normalized curves apply the
same learned final LayerNorm to each raw hidden state; they are not internal per-block pre-attention norms.

The historical normalized reference is **68.0**, the raw scale diagnostic is **150.0**, and the observed
Giant raw gap cutoff is **104.6**. Full precision is retained in config and machine-readable data.
A statistical mean+3σ tail or absolute reference exceedance is not automatically an artifact population.

In [ ]:
show_results(results)

## A — Raw and normalized distributions

Giant has a separated raw high-norm population that becomes a normalized low-norm population after final LayerNorm.
All 442 raw-gap patches are within a 443-token normalized low-norm group (split 37.7). The earlier Large normalized 21-versus-zero result is unchanged. All histogram y-axes are logarithmic.

In [ ]:
show_figures(results, [f'token_norm_histogram_{size}_{space}'
                       for size in ['base', 'large', 'giant'] for space in ['raw', 'normalized']])

## B — Layers, CLS and four separate registers

Layer cutoffs use each model/layer/space's own mean + 3×population std. Layer 0 is the embedding.
Prefix figures place patch distributions beside CLS and all four register trajectories. Prefix norms
never enter patch statistics. A larger register-model 3σ fraction is not proof of more isolated artifacts.

In [ ]:
show_figures(results, ['high_norm_across_layers_base', 'high_norm_across_layers_large',
                       'high_norm_across_layers_giant', 'prefix_token_norms_base',
                       'prefix_token_norms_large', 'prefix_token_norms_giant'])

## C — Same six visual inputs, both spaces

Raw and normalized norms and PCA use their named tensors. Paired norm maps share linear scales;
PCA fits are independent. Final CLS attention and full-token rollout are feature-space independent and
share log probability scales. Cosine maps use the corresponding raw or normalized features.

In [ ]:
show_figures(results, ['main_comparison'] +
                      [f'visual_comparison_{size}_{space}' for size in ['base', 'large', 'giant']
                       for space in ['raw', 'normalized']] +
                      ['featlens_comparison_giant_raw', 'featlens_comparison_giant_normalized'])

## Investigating Large's raw reversal

All 216 raw register-model patches above the pooled 3σ cutoff occur in the airplanes/bridge/sky image.
None exceeds that image's own 3σ cutoff. Global median rescaling leaves the pooled counts unchanged.
This is an image-mixture/rule effect, not prefix leakage or evidence of more paper-style isolated artifacts.
The 36 raw no-register tail patches also do not overlap the 21 normalized-reference outliers.

In [ ]:
show_figures(results, ['large_raw_tail_investigation'])

## Image-bootstrap location uncertainty and limitations

The COCO mask complement is a background proxy, not semantic ground truth. Seeded **1,000 whole-image
resamples** estimate a pooled outside/selected-token fraction and a 95% percentile interval. No selected
tokens means an undefined fraction and interval; zero-denominator bootstrap draws are counted explicitly.

Only 20 fixed images, three model sizes, one resolution, no retraining or seed/checkpoint sweep.
Normalization and threshold rules materially affect interpretation. Separately trained checkpoints,
attention/PCA visualizations, and mask complements do not establish a causal training mechanism.

See `README.md` for actual results and `docs/methods.md` for full definitions, prefix tables, the Large
investigation, bootstrap accounting, and explanations of what changed from the earlier study.
All numerical CSV/JSON tables and 300 dpi PNG/PDF figures are regenerated under `results/`.